# Learning Theory — VC Dimension

## Learning Objectives
- Understand why the finite-$k$ analysis breaks down for hypothesis classes parameterised by real numbers
- Define **shattering** and **VC dimension** $\text{VC}(\mathcal{H})$
- Show that the VC dimension of linear classifiers in $\mathbb{R}^n$ is $n+1$
- State the **Vapnik theorem**: uniform convergence holds when $\text{VC}(\mathcal{H}) < \infty$
- Understand why sample complexity is **linear in the VC dimension** (and thus in model parameters)

## Problem Statement

The finite-$\mathcal{H}$ analysis required $|\mathcal{H}| = k < \infty$. But any hypothesis class parameterised by real numbers — including linear classifiers — contains infinitely many functions.

The **VC dimension** provides a combinatorial notion of complexity that replaces the count $k$.

**Key theorem (Vapnik):**  
Let $d = \text{VC}(\mathcal{H})$. With probability at least $1 - \delta$, for all $h \in \mathcal{H}$:

$$|\varepsilon(h) - \hat{\varepsilon}(h)| \leq O\!\left(\sqrt{\frac{d}{m}\log\frac{m}{d} + \frac{1}{m}\log\frac{1}{\delta}}\right)$$

**Sample complexity corollary:**
$$m = O_{\gamma,\delta}(d)$$

The number of training examples needed is **linear** in the VC dimension — and for most hypothesis classes the VC dimension is linear in the number of parameters.

## 1. Shattering and VC Dimension

**Definition (Shattering).** $\mathcal{H}$ **shatters** a set $S = \{x^{(1)}, \ldots, x^{(d)}\}$ if for every labelling $\{y^{(1)}, \ldots, y^{(d)}\} \in \{0,1\}^d$, there exists some $h \in \mathcal{H}$ with $h(x^{(i)}) = y^{(i)}$ for all $i$.

**Definition (VC Dimension).** $\text{VC}(\mathcal{H})$ is the size of the **largest set** that $\mathcal{H}$ can shatter.

**Example — Linear classifiers in 2D:**
- $\mathcal{H} = \{h_\theta : x \mapsto \mathbf{1}\{\theta_0 + \theta_1 x_1 + \theta_2 x_2 \geq 0\}\}$
- Can shatter **any** 3 non-collinear points → $\text{VC}(\mathcal{H}) \geq 3$
- Cannot shatter **any** set of 4 points → $\text{VC}(\mathcal{H}) = 3$

**General result:** $\text{VC}(\mathcal{H}_{\text{linear}}) = n + 1$ for $\mathbb{R}^n$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

# Visualise all 8 labellings of 3 non-collinear points shattered by linear classifiers

# 3 non-collinear points
pts = np.array([[0.3, 0.7], [0.7, 0.8], [0.5, 0.2]])

def find_separator(pts, labels):
    """Return (w, b) of a linear separator for labels in {0,1} if one exists."""
    from sklearn.svm import SVC
    y = np.array(labels) * 2 - 1
    if len(np.unique(y)) == 1:
        # All same class — trivially separated
        return np.array([0.0, 1.0]), -0.5 * (np.max(pts[:,1]) + np.min(pts[:,1]))
    clf = SVC(kernel='linear', C=1000)
    clf.fit(pts, y)
    return clf.coef_[0], clf.intercept_[0]

labellings = list(product([0, 1], repeat=3))

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flatten()

x_plot = np.linspace(-0.1, 1.1, 200)

for ax, labels in zip(axes, labellings):
    labels = list(labels)
    w, b   = find_separator(pts, labels)

    # Draw separator
    if abs(w[1]) > 1e-6:
        y_line = -(w[0] * x_plot + b) / w[1]
        mask   = (y_line >= -0.2) & (y_line <= 1.2)
        ax.plot(x_plot[mask], y_line[mask], 'k-', lw=2)
    else:
        ax.axvline(-b / w[0], color='k', lw=2)

    # Plot points
    for i, (pt, lbl) in enumerate(zip(pts, labels)):
        if lbl == 1:
            ax.scatter(*pt, marker='x', s=120, c='#2166ac', linewidths=2.5, zorder=5)
        else:
            ax.scatter(*pt, marker='o', s=100, facecolors='none',
                       edgecolors='#d6604d', linewidths=2.5, zorder=5)

    lbl_str = ''.join(['×' if l == 1 else '○' for l in labels])
    ax.set_title(f'Labels: {lbl_str}', fontsize=9)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

fig.suptitle(
    'All $2^3 = 8$ labellings of 3 non-collinear points shattered by 2D linear classifiers\n'
    '→ VC(H_linear in 2D) ≥ 3',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

## 2. 4 Points Cannot Be Shattered

No set of 4 points in $\mathbb{R}^2$ can be shattered by a linear classifier.  
This is because any 4 points contain a convex subset whose XOR labelling cannot be linearly separated.

Therefore $\text{VC}(\mathcal{H}_{\text{linear in 2D}}) = 3$.

Note: to prove $\text{VC}(\mathcal{H}) \geq d$ we only need to show **one** set of size $d$ that is shattered — we do not need every set of size $d$ to be shatterable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

# 4 points in general position (convex hull)
pts4 = np.array([[0.2, 0.5], [0.8, 0.5], [0.5, 0.2], [0.5, 0.8]])

# XOR-like labelling: checkerboard — cannot be linearly separated
hard_labels_4pt = np.array([1, 1, -1, -1])          # separable
xor_labels      = np.array([1, -1, 1, -1])           # NOT separable

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x_plot = np.linspace(-0.1, 1.1, 200)
xx, yy = np.meshgrid(np.linspace(-0.1, 1.1, 200), np.linspace(-0.1, 1.1, 200))

# Left: separable labelling of 4 points
ax = axes[0]
clf = SVC(kernel='linear', C=1000)
clf.fit(pts4, hard_labels_4pt)
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.15)
ax.contour(xx, yy, Z, levels=[0], colors='k', linewidths=2)

for pt, lbl in zip(pts4, hard_labels_4pt):
    if lbl == 1:
        ax.scatter(*pt, marker='x', s=150, c='#2166ac', linewidths=3, zorder=5)
    else:
        ax.scatter(*pt, marker='o', s=130, facecolors='none',
                   edgecolors='#d6604d', linewidths=3, zorder=5)

acc = clf.score(pts4, hard_labels_4pt)
ax.set_title(f'4 points — separable labelling (acc={acc:.0f})\nLinear classifier exists ✓', fontsize=10)
ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

# Right: XOR labelling of 4 points — no linear separator
ax = axes[1]
clf2 = SVC(kernel='linear', C=1000)
clf2.fit(pts4, xor_labels)
Z2   = clf2.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z2, levels=20, cmap='RdBu', alpha=0.15)
ax.contour(xx, yy, Z2, levels=[0], colors='k', linewidths=2, linestyles='--')

for pt, lbl in zip(pts4, xor_labels):
    if lbl == 1:
        ax.scatter(*pt, marker='x', s=150, c='#2166ac', linewidths=3, zorder=5)
    else:
        ax.scatter(*pt, marker='o', s=130, facecolors='none',
                   edgecolors='#d6604d', linewidths=3, zorder=5)

# Show misclassified points
preds = clf2.predict(pts4)
_mis_labeled = False
for pt, lbl, pred in zip(pts4, xor_labels, preds):
    if lbl != pred:
        ax.scatter(*pt, s=400, marker='*', c='orange', zorder=6,
                   label=None if _mis_labeled else 'Misclassified')
        _mis_labeled = True

acc2 = clf2.score(pts4, xor_labels)
ax.set_title(f'4 points — XOR labelling (best linear acc={acc2:.2f})\nNo linear separator exists ✗', fontsize=10)
ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
if _mis_labeled:
    ax.legend(fontsize=9)

fig.suptitle('VC(H_linear in 2D) = 3 — 4 points cannot always be shattered',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Vapnik's Theorem and Sample Complexity

**Theorem (Vapnik).** Let $\mathcal{H}$ be given and $d = \text{VC}(\mathcal{H})$. With probability at least $1 - \delta$, for all $h \in \mathcal{H}$:

$$|\varepsilon(h) - \hat{\varepsilon}(h)| \leq O\!\left(\sqrt{\frac{d}{m}\log\frac{m}{d} + \frac{1}{m}\log\frac{1}{\delta}}\right)$$

**Corollary.** For $|\varepsilon(h) - \hat{\varepsilon}(h)| \leq \gamma$ to hold for all $h \in \mathcal{H}$ with probability at least $1-\delta$:
$$m = O_{\gamma,\delta}(d)$$

This is the most important theorem in learning theory. It tells us:
- **If $\text{VC}(\mathcal{H}) < \infty$**, uniform convergence occurs as $m \to \infty$: the algorithm will learn
- **Sample complexity is linear in $d$**: a $d$-parameter model needs $O(d)$ training examples
- **This validates the practitioner's rule of thumb**: collect $\sim 10\times$ the number of parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Plot the Vapnik bound as a function of m for various VC dimensions
# Using simplified form: O(sqrt(d/m * log(m/d)))
# Compared to the finite-H bound: O(sqrt(log(k)/m))

delta  = 0.05

def vapnik_bound(m, d, delta, C=1.0):
    """Simplified Vapnik bound term."""
    ratio = np.maximum(m / d, 1.01)
    return C * np.sqrt((d / m) * np.log(ratio) + (1 / m) * np.log(1 / delta))

def finite_h_bound(m, k, delta):
    """Notes4 §3 bound for finite |H|=k."""
    return np.sqrt((1 / (2 * m)) * np.log(2 * k / delta))

m_values = np.logspace(1, 4, 200).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: Vapnik bound vs m for different VC dimensions
ax = axes[0]
for d, color in [(2, '#2166ac'), (5, '#4dac26'), (10, '#d6604d'), (50, '#7b2d8b')]:
    bounds = vapnik_bound(m_values, d, delta)
    ax.loglog(m_values, bounds, '-', color=color, lw=2.5, label=f'$d={d}$')

ax.set_xlabel('Sample size $m$'); ax.set_ylabel('Generalisation gap bound')
ax.set_title(f'Vapnik Bound vs. $m$ for Various VC Dimensions\n($\\delta={delta}$)')
ax.legend(fontsize=9)
ax.text(0.03, 0.03,
        'Each curve decays as $O(\\sqrt{d/m})$\nHigher VC dim → slower convergence',
        transform=ax.transAxes, fontsize=8.5,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# Right: required m vs VC dimension to achieve bound ≤ γ
ax = axes[1]
d_values = np.arange(1, 101)

for gamma_target, ls in [(0.05, '-'), (0.10, '--'), (0.20, ':')]:
    m_req = []
    for d in d_values:
        # Binary search for smallest m where bound ≤ gamma_target
        lo, hi = d + 1, 100000
        for _ in range(40):
            mid = (lo + hi) // 2
            if vapnik_bound(mid, d, delta) <= gamma_target:
                hi = mid
            else:
                lo = mid + 1
        m_req.append(hi)
    ax.plot(d_values, m_req, ls, lw=2.5, label=f'$\\gamma={gamma_target}$')

# Reference: linear line
ax.plot(d_values, d_values * 20, 'k:', lw=1.5, alpha=0.5, label='$20d$ (reference)')

ax.set_xlabel('VC dimension $d$'); ax.set_ylabel('Required sample size $m$')
ax.set_title(f'Sample Complexity: $m = O(d)$ linear in VC dimension\n($\\delta={delta}$)')
ax.legend(fontsize=9)
ax.text(0.03, 0.97,
        'For most hypothesis classes\nVC dim ≈ number of parameters\n→ need ~O(params) training examples',
        transform=ax.transAxes, fontsize=8.5, va='top',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle("Vapnik's Theorem — $m = O_{\\gamma,\\delta}(d)$: Sample Complexity Linear in VC Dimension",
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. VC Dimension of Common Hypothesis Classes

| Hypothesis class | VC dimension |
|---|---|
| Linear classifiers in $\mathbb{R}^n$ | $n + 1$ |
| Axis-aligned rectangles in $\mathbb{R}^2$ | $4$ |
| Intervals on $\mathbb{R}$ | $2$ |
| Neural network with $W$ weights | $O(W \log W)$ |
| $k$-nearest neighbours | $\infty$ (not PAC learnable in finite data) |

For linear classifiers: VC = $n+1$ because
- Any $n+1$ points in general position (not lying on a hyperplane) can be shattered
- No set of $n+2$ points can be shattered (by Radon's theorem)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from sklearn.svm import SVC

def can_shatter(pts, n_labels=None):
    """Check whether a linear classifier can realise all 2^|pts| labellings."""
    n = len(pts)
    if n_labels is None:
        n_labels = 2 ** n
    for labels in product([-1, 1], repeat=n):
        y = np.array(labels)
        if len(np.unique(y)) == 1:
            continue
        clf = SVC(kernel='linear', C=1e6)
        clf.fit(pts, y)
        if clf.score(pts, y) < 1.0:
            return False
    return True

np.random.seed(0)

# Check: in d dimensions, can we shatter d+1 points (in general position)?
results = []
for dim in [1, 2, 3]:
    for n_pts in [dim, dim+1, dim+2]:
        pts = np.random.randn(n_pts, dim)
        shattered = can_shatter(pts)
        results.append((dim, n_pts, shattered))

fig, ax = plt.subplots(figsize=(9, 5))
ax.axis('off')

headers = ['Dimension $n$', 'Points $d$', 'Shattered?', 'Relation to VC = n+1']
table_data = []
for dim, n_pts, shattered in results:
    vc = dim + 1
    if n_pts < vc:
        relation = f'$d < \\text{{VC}} = {vc}$ → expect ✓'
    elif n_pts == vc:
        relation = f'$d = \\text{{VC}} = {vc}$ → expect ✓'
    else:
        relation = f'$d > \\text{{VC}} = {vc}$ → expect ✗'
    icon = '✓' if shattered else '✗'
    table_data.append([str(dim), str(n_pts), icon, relation])

tbl = ax.table(
    cellText=table_data, colLabels=headers,
    cellLoc='center', loc='center',
    bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)

for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#cce5ff')
        cell.set_text_props(fontweight='bold')
    elif table_data[row-1][2] == '✓':
        cell.set_facecolor('#d4edda')
    else:
        cell.set_facecolor('#f8d7da')

ax.set_title(
    'VC Dimension Verification: Linear classifiers in $\\mathbb{R}^n$ have VC dim = $n+1$\n'
    '(empirically verified by checking all $2^d$ labellings)',
    fontsize=11, pad=20
)
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="464"
     viewBox="0 0 780 464" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Infinite hypothesis class -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Infinite hypothesis</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">class &#x1d4d7;</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >|&#x1d4d7;| = &#x221e; for any real-parameter class &#x2014; finite-k analysis breaks down</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >combinatorial measure: can &#x1d4d7; realise every labelling</text>
  <text x="114" y="96" font-size="11.5" font-style="italic" fill="#555"
        >of d points? &#x2192; shattering</text>

  <!-- Row 2: VC dimension -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">VC(&#x1d4d7;) = d</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >largest set &#x1d4d7; can shatter &#x2014; finite if &#x1d4d7; has finite complexity</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >replace log|&#x1d4d7;| with VC(&#x1d4d7;) in</text>
  <text x="114" y="196" font-size="11.5" font-style="italic" fill="#555"
        >uniform convergence bound</text>

  <!-- Row 3: Uniform convergence bound -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Uniform</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">convergence</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >|&#x3b5;(h)&#x2212;&#x3b5;&#x302;(h)| &#x2264; O(&#x221a;(d/m &#xb7; log(m/d) + log(1/&#x3b4;)/m)) for all h &#x2208; &#x1d4d7;</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >solve for m: set bound = &#x3b3;</text>

  <!-- Row 4: Sample complexity -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Sample</text>
  <text x="102" y="352" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">complexity</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >m = O(d) &#x2014; linear in VC dimension, logarithmic refinements ignored</text>

  <!-- step 4&#x2192;5 -->
  <line x1="102" y1="358" x2="102" y2="408"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>

  <!-- Row 5: PAC learning -->
  <rect x="10" y="412" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="440" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">PAC learning</text>
  <line x1="197" y1="435" x2="216" y2="435"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="412" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="440" font-size="13" text-anchor="middle" fill="#333"
        >VC(&#x1d4d7;) &lt; &#x221e; &#x2192; algorithm learns from O(d) examples w.p. &#x2265; 1&#x2212;&#x3b4;</text>
</svg>
""")

## Summary

| Concept | Definition / Formula | Key Insight |
|---|---|---|
| Shattering | $\mathcal{H}$ realises all $2^d$ labellings of $d$ points | Measures expressive power combinatorially |
| VC dimension | $\text{VC}(\mathcal{H}) =$ size of largest shattered set | Replaces $\log\lvert\mathcal{H}\rvert$ for infinite classes |
| Linear classifiers in $\mathbb{R}^n$ | $\text{VC} = n + 1$ | Matches number of parameters (plus bias) |
| Vapnik bound | $\lvert\varepsilon(h)-\hat{\varepsilon}(h)\rvert \leq O\!\left(\sqrt{\frac{d}{m}\log\frac{m}{d}}\right)$ | Uniform convergence when $\text{VC}(\mathcal{H}) < \infty$ |
| Sample complexity | $m = O_{\gamma,\delta}(d)$ | Linear in VC dimension; validates 10× parameters rule |
| PAC learnability | $\text{VC}(\mathcal{H}) < \infty$ iff $\mathcal{H}$ is PAC learnable | VC dimension characterises what can be learned |

**Key insight:** VC dimension is the right notion of complexity for real-parameter hypothesis classes — it replaces the logarithm of the class size and shows that $O(d)$ examples suffice for a class of VC dimension $d$.